# Sequence-модель: окно 30–60 баров

**Шаг 4 экспериментов.** Окна 30 и 60 для учёта истории. Rolling stats по окну вместо LSTM (избегаем OOM).

**Сравнение с рабочим пайплайном:** данные и сплит как в 07/08 — temporal split (train 8 дней, val 1, test последний день), бектест только на тестовом дне с той же логикой комиссий и HOLD (hold_keep_prev). Так можно честно сравнить AUC и net PnL: даёт ли добавление rolling 30/60 прирост относительно baseline.

## 1. Импорты и подготовка

In [7]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import joblib

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)
import lightgbm as lgb

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(labeled_path):
    raise FileNotFoundError('Запустите 04_Data_Labeling_And_Feature_Loading.ipynb')
if not os.path.exists(feat_path):
    raise FileNotFoundError('Запустите 06_Feature_Selection.ipynb')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]

print(f'Строк: {len(df):,}, target=tp_sl_1_05 (колонка target)')

Строк: 344,415, target=tp_sl_1_05 (колонка target)


## 2. Rolling-фичи (окна 30, 60)

In [8]:
key_feats = BASE_FEATURES[:10]
for w in [30, 60]:
    grp = df.groupby('session_key', group_keys=False)
    for c in key_feats:
        if c in df.columns:
            df[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_COLS = [c for c in df.columns if '_roll' in c]
FEATURES_SEQ = BASE_FEATURES + SEQ_COLS
FEATURES_SEQ = [c for c in FEATURES_SEQ if c in df.columns]
print(f'Фичей: baseline {len(BASE_FEATURES)}, +rolling {len(FEATURES_SEQ)}')

Фичей: baseline 22, +rolling 62


## 3. Обучение и бектест

In [9]:
# Как в рабочем пайплайне (07/08): temporal split по датам, бектест только на последнем дне
valid = df.dropna(subset=FEATURES_SEQ + ['target']).copy()
valid = valid[valid['target'].isin([-1.0, 1.0])]
valid['y'] = (valid['target'] == 1).astype(int)
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
if len(dates) < 10:
    raise ValueError(f'Мало дат для temporal split: {len(dates)}. Нужно минимум 10 дней.')
train_dates = set(dates[:8])
val_dates = set([dates[8]])
test_dates = set([dates[9]])
train_df = valid[valid['date'].isin(train_dates)]
val_df = valid[valid['date'].isin(val_dates)]
test_df = valid[valid['date'].isin(test_dates)]

# Порог как в прод-модели (best_model.joblib)
THR = 0.45
COMMISSION_RT = 0.001

def backtest_pnl(proba, ret, session_ids, threshold=THR, commission_rt=COMMISSION_RT, hold_keep_prev=True):
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev if hold_keep_prev else 0.0
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total = pos_changed.sum() * (commission_rt / 2.0)
    pnl_net = (pos * ret).sum() - fee_total
    return {'trades': int(pos_changed.sum()), 'net_%': float(pnl_net * 100)}

def run_temporal(features):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[features].fillna(0))
    X_val = scaler.transform(val_df[features].fillna(0))
    X_test = scaler.transform(test_df[features].fillna(0))
    # DataFrame с именами колонок, чтобы LGBM не выдавал предупреждение про feature names
    X_train_df = pd.DataFrame(X_train, columns=features)
    X_val_df = pd.DataFrame(X_val, columns=features)
    X_test_df = pd.DataFrame(X_test, columns=features)
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
    model.fit(X_train_df, train_df['y'].values)
    val_auc = roc_auc_score(val_df['y'].values, model.predict_proba(X_val_df)[:, 1])
    test_auc = roc_auc_score(test_df['y'].values, model.predict_proba(X_test_df)[:, 1])
    proba_bt = model.predict_proba(X_test_df)[:, 1]
    return val_auc, test_auc, proba_bt

ret = test_df['ret_next'].values
sess = test_df['session_key'].values

print('Temporal split: train=', f'{min(train_dates)}..{max(train_dates)}', f'({len(train_df):,}) | val={dates[8]} ({len(val_df):,}) | test={dates[9]} ({len(test_df):,})')
print()

val_auc_b, test_auc_b, proba_b = run_temporal(BASE_FEATURES)
val_auc_s, test_auc_s, proba_s = run_temporal(FEATURES_SEQ)

# Зоны HOLD: 0.45-0.55 (tuned), 0.40-0.60 (прод), 0.35-0.65, 0.30-0.70, 0.25-0.75
bt_b_45 = backtest_pnl(proba_b, ret, sess, threshold=0.45)
bt_b_60 = backtest_pnl(proba_b, ret, sess, threshold=0.60)
bt_b_65 = backtest_pnl(proba_b, ret, sess, threshold=0.65)
bt_b_70 = backtest_pnl(proba_b, ret, sess, threshold=0.70)
bt_b_75 = backtest_pnl(proba_b, ret, sess, threshold=0.75)
bt_s_45 = backtest_pnl(proba_s, ret, sess, threshold=0.45)
bt_s_60 = backtest_pnl(proba_s, ret, sess, threshold=0.60)
bt_s_65 = backtest_pnl(proba_s, ret, sess, threshold=0.65)
bt_s_70 = backtest_pnl(proba_s, ret, sess, threshold=0.70)
bt_s_75 = backtest_pnl(proba_s, ret, sess, threshold=0.75)

print('Baseline (только выбранные фичи):')
print(f'  Val AUC={val_auc_b:.4f}, Test AUC={test_auc_b:.4f}')
print(f'  0.45-0.55:      net PnL={bt_b_45["net_%"]:.2f}%, trades={bt_b_45["trades"]}')
print(f'  0.40-0.60 (прод): net PnL={bt_b_60["net_%"]:.2f}%, trades={bt_b_60["trades"]}')
print(f'  0.35-0.65:      net PnL={bt_b_65["net_%"]:.2f}%, trades={bt_b_65["trades"]}')
print(f'  0.30-0.70:      net PnL={bt_b_70["net_%"]:.2f}%, trades={bt_b_70["trades"]}')
print(f'  0.25-0.75:      net PnL={bt_b_75["net_%"]:.2f}%, trades={bt_b_75["trades"]}')
print('Sequence (baseline + rolling 30, 60):')
print(f'  Val AUC={val_auc_s:.4f}, Test AUC={test_auc_s:.4f}')
print(f'  0.45-0.55:      net PnL={bt_s_45["net_%"]:.2f}%, trades={bt_s_45["trades"]}')
print(f'  0.40-0.60 (прод): net PnL={bt_s_60["net_%"]:.2f}%, trades={bt_s_60["trades"]}')
print(f'  0.35-0.65:      net PnL={bt_s_65["net_%"]:.2f}%, trades={bt_s_65["trades"]}')
print(f'  0.30-0.70:      net PnL={bt_s_70["net_%"]:.2f}%, trades={bt_s_70["trades"]}')
print(f'  0.25-0.75:      net PnL={bt_s_75["net_%"]:.2f}%, trades={bt_s_75["trades"]}')
print()
print('Прирост от rolling 30/60:')
print('  0.45-0.55:   Delta Test AUC =', f'{test_auc_s - test_auc_b:+.4f}', ', Delta net PnL =', f'{bt_s_45["net_%"] - bt_b_45["net_%"]:+.2f}%', ', Delta trades =', bt_s_45["trades"] - bt_b_45["trades"])
print('  0.40-0.60:   Delta net PnL =', f'{bt_s_60["net_%"] - bt_b_60["net_%"]:+.2f}%', ', Delta trades =', bt_s_60["trades"] - bt_b_60["trades"])
print('  0.35-0.65:   Delta net PnL =', f'{bt_s_65["net_%"] - bt_b_65["net_%"]:+.2f}%', ', Delta trades =', bt_s_65["trades"] - bt_b_65["trades"])
print('  0.30-0.70:   Delta net PnL =', f'{bt_s_70["net_%"] - bt_b_70["net_%"]:+.2f}%', ', Delta trades =', bt_s_70["trades"] - bt_b_70["trades"])
print('  0.25-0.75:   Delta net PnL =', f'{bt_s_75["net_%"] - bt_b_75["net_%"]:+.2f}%', ', Delta trades =', bt_s_75["trades"] - bt_b_75["trades"])

Temporal split: train= 2026-02-01..2026-02-08 (146,693) | val=2026-02-09 (24,014) | test=2026-02-10 (15,356)

Baseline (только выбранные фичи):
  Val AUC=0.7646, Test AUC=0.6265
  0.45-0.55:      net PnL=889.29%, trades=4864
  0.40-0.60 (прод): net PnL=932.74%, trades=1705
  0.35-0.65:      net PnL=813.94%, trades=1038
  0.30-0.70:      net PnL=701.40%, trades=636
  0.25-0.75:      net PnL=516.47%, trades=367
Sequence (baseline + rolling 30, 60):
  Val AUC=0.7682, Test AUC=0.6291
  0.45-0.55:      net PnL=934.35%, trades=4619
  0.40-0.60 (прод): net PnL=934.89%, trades=1492
  0.35-0.65:      net PnL=824.45%, trades=910
  0.30-0.70:      net PnL=620.04%, trades=529
  0.25-0.75:      net PnL=573.61%, trades=276

Прирост от rolling 30/60:
  0.45-0.55:   Delta Test AUC = +0.0026 , Delta net PnL = +45.06% , Delta trades = -245
  0.40-0.60:   Delta net PnL = +2.15% , Delta trades = -213
  0.35-0.65:   Delta net PnL = +10.51% , Delta trades = -128
  0.30-0.70:   Delta net PnL = -81.36% , Delt

**Про предупреждения (X does not have valid feature names):** LightGBM при обучении на DataFrame запоминает имена фичей; при вызове `predict_proba` на NumPy-массиве (после `scaler.transform`) имён нет — sklearn выдаёт предупреждение. На предсказания оно не влияет. В коде выше после масштабирования данные оборачиваются в `pd.DataFrame(..., columns=features)`, чтобы в модель передавались имена колонок и предупреждение не появлялось.

**Что видно по итогам:**
- Сравниваются **пять зон HOLD**: 0.45–0.55, 0.40–0.60 (прод), 0.35–0.65, 0.30–0.70, 0.25–0.75. Чем шире зона, тем меньше сделок; по цифрам видно, как меняется net PnL (часто сначала растёт, потом падает при очень широкой зоне).
- **Sequence (rolling 30, 60)** при любой зоне даёт прирост по Test AUC и по net PnL и меньше сделок относительно baseline.
- **Итог:** по таблице и выводу выше можно выбрать зону и набор фичей для прода — см. раздел «Какую модель/стратегию стоит попробовать в проде» ниже.

## Итог шага 11 (Sequence 30–60). Зафиксированные выводы

**Условия эксперимента:**
- Rolling mean/std по окнам 30 и 60 добавлены к первым 10 базовым фичам (всего 62 фичи). Baseline — только 22 фичи из `features_selected_tp_sl_1_05.txt`.
- Данные и сплит как в рабочем пайплайне: temporal split (8 дней train, 1 val, 1 test), бектест только на последнем дне (test date), комиссия 0.1% round-trip, HOLD = сохранение позиции.

**Что такое зоны HOLD:** Задаётся thr_hi; thr_lo = 1 − thr_hi. **BUY** при proba ≥ thr_hi, **SELL** при proba ≤ thr_lo, **HOLD** в интервале (позиция сохраняется). В таблице и выводе везде пишем зону явно, чтобы не путаться.
- **0.45–0.55** — узкая зона (подобрано по val F1 в шаге 07). Много сделок.
- **0.40–0.60** — прод (model_wide_hold_040_060). Меньше сделок, выше PnL на тесте.
- **0.35–0.65**, **0.30–0.70**, **0.25–0.75** — ещё шире: меньше сделок; на бектесте PnL может падать, но в реале меньше оборотов может быть плюсом (исполнение, проскальзывание, нагрузка).

**Результаты (temporal split: train 2026-02-01..08, val 2026-02-09, test 2026-02-10):**

| Зона HOLD         | Baseline (net PnL, trades) | Sequence (net PnL, trades) | Delta PnL | Delta trades |
|-------------------|---------------------------|----------------------------|-----------|--------------|
| **0.45–0.55**     | 889.29%, 4864             | 934.35%, 4619              | +45.06%   | −245         |
| **0.40–0.60** (прод) | 932.74%, 1705          | 934.89%, 1492              | +2.15%    | −213         |
| **0.35–0.65**     | 813.94%, 1038             | 824.45%, 910               | +10.51%   | −128         |
| **0.30–0.70**     | 701.40%, 636              | 620.04%, 529               | −81.36%   | −107         |
| **0.25–0.75**     | 516.47%, 367              | 573.61%, 276               | +57.13%   | −91          |

Метрики: Baseline Val AUC=0.7646, Test AUC=0.6265; Sequence Val AUC=0.7682, Test AUC=0.6291 (прирост Test AUC +0.0026).

**Выводы:**
1. **Максимум net PnL** — у зоны **0.40–0.60** (932% baseline, 935% sequence). При расширении зоны PnL падает: 0.35–0.65 → 814% / 824%; 0.30–0.70 → 701% / 620%; 0.25–0.75 → 516% / 574%.
2. **Число сделок** монотонно убывает с расширением зоны: 4864 → 1705 → 1038 → 636 → 367 (baseline); 4619 → 1492 → 910 → 529 → 276 (sequence).
3. **Sequence vs Baseline:** При большинстве зон Sequence даёт выше PnL и меньше сделок. **Исключение — 0.30–0.70**: Baseline 701% > Sequence 620% (−81% Delta). При 0.25–0.75 Sequence снова выше (574% vs 516%). При очень широких зонах относительный эффект rolling-фичей нестабилен.
4. **Компромисс PnL / trades:** Зона **0.40–0.60** даёт лучший баланс (максимальный PnL при 1492/1705 сделках). **0.35–0.65** — меньше сделок (910/1038), PnL ниже; **0.30–0.70** и **0.25–0.75** — ещё меньше сделок, но PnL падает сильнее. В реале меньше сделок может окупаться за счёт исполнения и риска — варианты 0.35–0.65 и 0.30–0.70 можно пробовать в проде отдельно.

---

## Какую модель/стратегию стоит попробовать в проде

1. **Зона HOLD:** По тесту лидирует **0.40–0.60** (текущий прод). **0.35–0.65** — компромисс: меньше сделок (910/1038), PnL ниже. **0.30–0.70** и **0.25–0.75** — заметно меньше сделок (529/636 и 276/367), PnL падает сильнее. **До защиты** оставляем 0.40–0.60.

2. **Baseline vs Sequence:** Лучший PnL — **Sequence при 0.40–0.60** (934.89%, 1492). При 0.30–0.70 у Sequence PnL ниже Baseline (620 vs 701) — в этой зоне rolling-фичи не дают преимущества. При 0.25–0.75 Sequence выше (574 vs 516).

3. **Рекомендация:**
   - **До защиты:** baseline + 0.40–0.60 (model_wide_hold_040_060).
   - **После защиты:** (1) попробовать **Sequence + 0.40–0.60**; (2) при желании снизить оборот — протестировать в проде **0.35–0.65** или **0.30–0.70** (Sequence или baseline) и оценить по реальному исполнению; для 0.30–0.70 на бектесте baseline лучше sequence.